<a href="https://colab.research.google.com/github/anjicx/CNHypergraph/blob/main/Visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE_DIR: /content/drive/MyDrive/PatientData
STATISTICS_DIR: /content/drive/MyDrive/PatientData/hgnn_statistics
PLOTS_DIR: /content/drive/MyDrive/PatientData/plots
Plots will be saved to: /content/drive/MyDrive/PatientData/plots/descriptive_simple
OK: /content/drive/MyDrive/PatientData/hgnn_statistics/combined_basic_hypergraph_statistics.csv
OK: /content/drive/MyDrive/PatientData/hgnn_statistics/combined_multimorbidity_burden_groups.csv
OK: /content/drive/MyDrive/PatientData/hgnn_statistics/combined_rare_diagnosis_stats.csv
OK: /content/drive/MyDrive/PatientData/hgnn_statistics/combined_same_day_tie_summary.csv
OK: /content/drive/MyDrive/PatientData/hgnn_statistics/combined_lag_quality_summary.csv
OK: /content/drive/MyDrive/PatientData/hgnn_statistics/combined_target_imbalance_summary.csv
OK: /content/drive/MyDrive/PatientData/hgnn_statistics/KEY_SUMMARY_BY_S

In [ ]:
validation = pd.read_csv("combined_validation_summary.csv")
same_day = pd.read_csv("combined_same_day_tie_summary.csv")
lag_quality = pd.read_csv("combined_lag_quality_summary.csv")
target_imbalance = pd.read_csv("combined_target_imbalance_summary.csv")


In [ ]:
groups = ["Male", "Female"]

# HELPER FUNCTIONS FOR PLOTS

def group_path(group, filename):
    return f"{group.lower()}_{filename}"

def save_plot(filename):
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()
    print("saved:", filename)

def clean_axes(xlabel="", ylabel=""):
    ax = plt.gca()
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def plot_ccdf(values, label):
    values = pd.Series(values).replace([np.inf, -np.inf], np.nan).dropna()
    values = values[values > 0].sort_values().values

    if len(values) == 0:
        return

    y = 1.0 - np.arange(1, len(values) + 1) / len(values)
    plt.plot(values, y, marker=".", linestyle="none", markersize=3, label=label)

# TECHNICAL QC SUMMARY

These plots and tables support validation, temporal ordering checks, and prediction-task preparation.
Descriptive burden/prevalence plots are kept in `DescriptiveAnalysis.ipynb`.


In [ ]:
summary = (
    validation
    .merge(same_day, on="group", how="left")
    .merge(lag_quality, on="group", how="left")
    .merge(target_imbalance, on="group", how="left")
)

summary_cols = [
    "group",
    "rows",
    "patients",
    "missing_key_values",
    "duplicated_patient_diagnosis_memberships",
    "invalid_first_dates",
    "rows_before_observation_start",
    "wrong_hyperedge_ids",
    "patients_with_inconsistent_attributes",
    "pct_patients_with_same_day_tie",
    "pct_zero_lags",
    "median_lag_days",
    "n_prefix_samples",
    "n_unique_target_diagnoses",
    "most_common_target_share",
    "pct_targets_with_1_sample"
]

summary_table = summary[summary_cols].copy()
print(summary_table)
summary_table.to_csv("KEY_TECHNICAL_SUMMARY_FOR_REPORT.csv", index=False)


## Temporal descriptive statistics

**Question:** How do patient diagnosis histories change over time?

1. **Sequence length**  
   → How many unique diagnoses does a patient have in their ordered history?  
   This describes how complex the patient trajectory is.

2. **Follow-up years**  
   → How long is the patient observed between first and last diagnosis?  
   This is important because more diagnoses can appear simply because the patient is followed for longer.

3. **Diagnoses per year**  
   → How quickly do diagnoses accumulate over time?  
   This adjusts diagnosis burden by follow-up time.

4. **Lag days**  
   → How many days pass between one diagnosis and the next?  
   This describes the timing between consecutive diagnosis events.

5. **Zero-lag share**  
   → How often do consecutive diagnoses appear on the same date?  
   These should be interpreted carefully because same-day diagnoses do not have a clear temporal order.

6. **Same-day ties**  
   → How often do multiple diagnoses first appear on the same day?  
   This shows how often temporal ordering is ambiguous.

In [ ]:
sequence_data = []
labels = []

for group in groups:
    df = pd.read_csv(group_path(group, "patient_temporal_summary.csv"))
    sequence_data.append(df["sequence_length"].dropna())
    labels.append(group)

plt.figure(figsize=(7, 5))
plt.boxplot(sequence_data, labels=labels, showfliers=False)

clean_axes(
    xlabel="",
    ylabel="sequence length"
)
save_plot("08_sequence_length_boxplot.png")

saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/08_sequence_length_boxplot.png


In [ ]:
plt.figure(figsize=(8, 5))

for group in groups:
    df = pd.read_csv(group_path(group, "patient_temporal_summary.csv"))
    plt.hist(np.log1p(df["sequence_length"]), bins=40, alpha=0.6, label=group)

clean_axes(
    xlabel="log(1 + sequence length)",
    ylabel="patients"
)
plt.legend()
save_plot("09_sequence_length_log_hist.png")

saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/09_sequence_length_log_hist.png


In [ ]:
followup_data = []
labels = []

for group in groups:
    df = pd.read_csv(group_path(group, "patient_temporal_summary.csv"))
    followup_data.append(df["follow_up_years"].dropna())
    labels.append(group)

plt.figure(figsize=(7, 5))
plt.boxplot(followup_data, labels=labels, showfliers=False)

clean_axes(
    xlabel="",
    ylabel="follow-up years"
)
save_plot("10_follow_up_years_boxplot.png")

plt.figure(figsize=(8, 5))

for group in groups:
    df = pd.read_csv(group_path(group, "patient_temporal_summary.csv"))
    values = df["follow_up_years"].replace([np.inf, -np.inf], np.nan).dropna()
    plt.hist(values, bins=40, alpha=0.6, label=group)

clean_axes(
    xlabel="follow-up years",
    ylabel="patients"
)
plt.legend()
save_plot("11_follow_up_years_hist.png")

saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/10_follow_up_years_boxplot.png
saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/11_follow_up_years_hist.png


In [ ]:
plt.figure(figsize=(8, 5))

for group in groups:
    df = pd.read_csv(group_path(group, "patient_temporal_summary.csv"))
    values = df["diagnoses_per_year"].replace([np.inf, -np.inf], np.nan).dropna()
    values = values[values <= values.quantile(0.99)]

    plt.hist(values, bins=40, alpha=0.6, label=group)

clean_axes(
    xlabel="diagnoses per year",
    ylabel="patients"
)
plt.legend()
save_plot("12_diagnoses_per_year_hist.png")

saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/12_diagnoses_per_year_hist.png


In [ ]:
plt.figure(figsize=(8, 5))

for group in groups:
    df = pd.read_csv(group_path(group, "consecutive_transition_rows.csv"))
    values = df["lag_days"].replace([np.inf, -np.inf], np.nan).dropna()
    values = values[values >= 0]

    plt.hist(np.log1p(values), bins=40, alpha=0.6, label=group)

clean_axes(
    xlabel="log(1 + lag days)",
    ylabel="transitions"
)
plt.legend()
save_plot("13_lag_days_log_hist.png")
lag_quality = pd.read_csv("combined_lag_quality_summary.csv")

plt.figure(figsize=(6, 5))
plt.bar(lag_quality["group"], lag_quality["pct_zero_lags"])

clean_axes(
    xlabel="",
    ylabel="zero-lag share"
)
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
save_plot("14_zero_lag_share.png")


saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/13_lag_days_log_hist.png
saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/14_zero_lag_share.png


In [ ]:
same_day = pd.read_csv("combined_same_day_tie_summary.csv")

plt.figure(figsize=(6, 5))
plt.bar(same_day["group"], same_day["pct_patients_with_same_day_tie"])

clean_axes(
    xlabel="",
    ylabel="patients with same-day ties"
)
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
save_plot("15_same_day_ties.png")

saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/15_same_day_ties.png


PREDICTION-TASK STATISTICS

Question: What does the next-diagnosis prediction task look like?

1. Prefix length
   -> How many previous diagnoses are used to predict the next one?

2. Number of prefix samples
   -> How many prediction examples are available?

3. Target distribution
   -> Which diagnoses most often appear as the next diagnosis?

4. Most common target share
   -> Is the task dominated by one very frequent next diagnosis?

5. Rare targets
   -> How many next-diagnosis labels appear only once?

6. Target Pareto plot
   -> Do a few diagnoses cover most prediction samples?

Why these are useful:
They show whether the prediction task has enough samples, how many possible target diagnoses exist, and whether the labels are imbalanced.

In [ ]:
plt.figure(figsize=(8, 5))

for group in groups:
    df = pd.read_csv(group_path(group, "prefix_prediction_samples.csv"))
    plt.hist(np.log1p(df["prefix_length"]), bins=40, alpha=0.6, label=group)

clean_axes(
    xlabel="log(1 + prefix length)",
    ylabel="samples"
)
plt.legend()
save_plot("16_prefix_length_log_hist.png")

saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/16_prefix_length_log_hist.png


In [ ]:
for group in groups:
    df = pd.read_csv(group_path(group, "target_distribution_for_next_disease.csv"))
    df = df.head(15).sort_values("n_target_occurrences")

    plt.figure(figsize=(8, 6))
    plt.barh(df["icd_code"], df["n_target_occurrences"])

    clean_axes(
        xlabel="target occurrences",
        ylabel="ICD code"
    )
    save_plot(f"17_top_targets_{group.lower()}.png")

saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/17_top_targets_male.png
saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/17_top_targets_female.png


In [ ]:
for group in groups:
    df = pd.read_csv(group_path(group, "target_distribution_for_next_disease.csv"))
    df = df.sort_values("n_target_occurrences", ascending=False).copy()

    df["rank"] = np.arange(1, len(df) + 1)
    df["cumulative_share"] = (
        df["n_target_occurrences"].cumsum()
        / df["n_target_occurrences"].sum()
    )

    plt.figure(figsize=(8, 5))
    plt.plot(df["rank"], df["cumulative_share"])

    clean_axes(
        xlabel="target rank",
        ylabel="cumulative share"
    )
    plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    save_plot(f"18_target_pareto_{group.lower()}.png")

saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/18_target_pareto_male.png
saved: /content/drive/MyDrive/PatientData/plots/descriptive_simple/18_target_pareto_female.png


## Sex-age technical visualizations

These plots show temporal and prediction-preparation behaviour by sex and age group.
Descriptive sex-age disease burden plots are kept in `DescriptiveAnalysis.ipynb`.


In [ ]:
sex_age_summary = pd.read_csv(
    "KEY_SUMMARY_BY_SEX_AGE_FOR_REPORT.csv"
)

sex_age_summary["age_group"] = sex_age_summary["age_group"].astype(str).str.strip()
sex_age_summary["age_start"] = (
    sex_age_summary["age_group"]
    .str.extract(r"(\d+)")
    .astype(float)
)
sex_age_summary = sex_age_summary.sort_values(["sex", "age_start"])

print(sex_age_summary.head())


In [ ]:


def save_plot_age(filename):
    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()
    print("saved:", filename)

In [ ]:

# Median sequence length by sex and age group

plt.figure(figsize=(10, 5))

for sex in ["Male", "Female"]:
    df = sex_age_summary[sex_age_summary["sex"] == sex].copy()
    df = df.sort_values("age_start")

    plt.plot(
        df["age_group"],
        df["median_sequence_length"],
        marker="o",
        label=sex
    )

clean_axes(
    xlabel="age group",
    ylabel="median sequence length"
)

plt.xticks(rotation=45, ha="right")
plt.legend()
save_plot_age("A04_median_sequence_length_by_sex_age.png")

saved: /content/drive/MyDrive/PatientData/plots/by_sex_age/A04_median_sequence_length_by_sex_age.png


In [ ]:

# Median diagnoses per year by sex and age group


plt.figure(figsize=(10, 5))

for sex in ["Male", "Female"]:
    df = sex_age_summary[sex_age_summary["sex"] == sex].copy()
    df = df.sort_values("age_start")

    plt.plot(
        df["age_group"],
        df["median_diagnoses_per_year"],
        marker="o",
        label=sex
    )

clean_axes(
    xlabel="age group",
    ylabel="median diagnoses per year"
)

plt.xticks(rotation=45, ha="right")
plt.legend()
save_plot_age("A05_median_diagnoses_per_year_by_sex_age.png")

saved: /content/drive/MyDrive/PatientData/plots/by_sex_age/A05_median_diagnoses_per_year_by_sex_age.png


In [ ]:
# Rare targets by sex and age group

plt.figure(figsize=(10, 5))

for sex in ["Male", "Female"]:
    df = sex_age_summary[sex_age_summary["sex"] == sex].copy()
    df = df.sort_values("age_start")

    plt.plot(
        df["age_group"],
        df["pct_targets_with_1_sample"],
        marker="o",
        label=sex
    )

clean_axes(
    xlabel="age group",
    ylabel="share of targets with one sample"
)

plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.xticks(rotation=45, ha="right")
plt.legend()
save_plot_age("A07_rare_targets_by_sex_age.png")

saved: /content/drive/MyDrive/PatientData/plots/by_sex_age/A07_rare_targets_by_sex_age.png
